In [58]:
import os
import re
from datetime import datetime
import pandas as pd
import numpy as np
from plotnine import *
import baltic as bt
from geopy.distance import geodesic
import geopandas as gpd
import geodatasets
from shapely.geometry import Polygon, MultiPolygon
import json
import sys
import subprocess
import glob
import shutil
import importlib
import matplotlib as mpl

In [3]:
def load_tree(nexusPath):
    mytree = bt.loadNexus(nexusPath, tip_regex='\|([0-9]{4}\-[0-9]{2}\-[0-9]{2})', absoluteTime=True, verbose=False) # your tip regex might have to change or you could convert you tree into a newick by deleting everything before the tree string lol
    return(mytree)

In [38]:
# Load tree
nexusPath="./SERAPHIM/nov2021-2024_no-suliformes/NAm_wild-aves_nov2021-2024_no-suliformes_ha_contphyl_2025-08-18_MCC.tree" # your path here
mytree = load_tree(nexusPath)

In [39]:
print(type(mytree))

<class 'baltic.baltic.tree'>


In [40]:
print(mytree.treeStats())


Tree height: 3.325378
Tree length: 696.371663
strictly bifurcating tree
annotations present

Numbers of objects in tree: 10995 (5497 nodes and 5498 leaves)

None


In [41]:
#print(mytree.Objects[9].traits)

In [42]:
# Add a label for each object in mytree

label_count = 0

for k in mytree.Objects:
    k.label = label_count
    label_count += 1

In [43]:
print(mytree.Objects[3].parent)

thiscount = 0

for k in mytree.Objects:
    if thiscount < 1:
        if k.is_leaf():
            print(k.name)
            thiscount += 1

a/canadagoose/on/fav02822/2022|2022-04-11|wild|anseriformes|MississippiFlyway


In [118]:
def backwards(leaf, path_to_leaf, t, time):
    if time < t:
        path_to_leaf.append(leaf.parent)
        time += leaf.traits["length"]

        if leaf.parent.absoluteTime < 2021.7: # if parent is the root
            pass
        else:
            path_to_leaf = backwards(leaf.parent, path_to_leaf,t, time)
        
    return(path_to_leaf) # to add the latest node as a new "leaf", just print latest node and start afresh

In [119]:
count = 0
parents = {}

for k in mytree.Objects:
    t = 0.3 # 0.3 years is about 4 months
    if count < 10:
        if k.is_node():
            pass
        else:
            path_to_leaf = []
            time = 0
            path_to_leaf = backwards(k, path_to_leaf, t, time)
            # parents = defaultdict(path_to_leaf)
            # parents[k.name] = path_to_leaf
            parents.update({k:{"path":path_to_leaf}})
            count += 1

In [120]:
print(parents)

{<baltic.baltic.leaf object at 0x32ac94050>: {'path': [<baltic.baltic.node object at 0x32ac8ff90>, <baltic.baltic.node object at 0x32ab06cd0>, <baltic.baltic.node object at 0x32ac8ff50>, <baltic.baltic.node object at 0x32ac8ff10>, <baltic.baltic.node object at 0x32ac8fed0>]}, <baltic.baltic.leaf object at 0x32ac94850>: {'path': [<baltic.baltic.node object at 0x32ac947d0>, <baltic.baltic.node object at 0x32ac8ff90>, <baltic.baltic.node object at 0x32ab06cd0>]}, <baltic.baltic.leaf object at 0x32ac95090>: {'path': [<baltic.baltic.node object at 0x32ac947d0>, <baltic.baltic.node object at 0x32ac8ff90>, <baltic.baltic.node object at 0x32ab06cd0>, <baltic.baltic.node object at 0x32ac8ff50>, <baltic.baltic.node object at 0x32ac8ff10>, <baltic.baltic.node object at 0x32ac8fed0>, <baltic.baltic.node object at 0x32ac8fe90>]}, <baltic.baltic.leaf object at 0x32ac96ad0>: {'path': [<baltic.baltic.node object at 0x32ac96a10>, <baltic.baltic.node object at 0x32ab06cd0>, <baltic.baltic.node object at

In [121]:
# Calculate distance between leaf and each node

In [122]:
def calc_dist(node, leaf_lat, leaf_long):
    # For each node, pull the lat/long
    node_lat = node.traits["location1_median"]
    node_long = node.traits["location2_median"]

    # Establish coordinates
    start = (node_lat, node_long)
    end = (leaf_lat, leaf_long)

    # Calculate distance
    dist = geodesic(start, end, ellipsoid='GRS-80').kilometers

    return(dist)

In [123]:
# Calculate direction of movement

In [124]:
def calc_direction(node, leaf_lat, leaf_long):
    # For each node, pull the lat/long
    node_lat = node.traits["location1_median"]
    node_long = node.traits["location2_median"]

    # Calculate change in latitude (and set to direction)
    changeLat = float(leaf_lat - node_lat)
    # print(changeLat)
    # if changeLat < 0:
    #     directLat = "south"
    # elif changeLat > 0:
    #     directLat = "north"
    # else:
    #     directLat = "none"

    # Calculate change in longitude (and set to direction)
    changeLong = float(leaf_long - node_long)
    # print(changeLong)
    # if changeLong < 0:
    #     directLong = "west"
    # elif changeLong > 0:
    #     directLong = "east"
    # else:
    #     directLong = "none"

    # Combine
    return(changeLat, changeLong)
    # return(directLat_directLong)

In [125]:
for key in parents:
    # Update the parents dictionary
    parents[key]["distance"] = []
    parents[key]["changeLatitude"] = []
    parents[key]["changeLongitude"] = []

    # Initialize the lists
    distance = []
    changeLat = []
    changeLong = []

    # Set leaf (ending) latitude and longitude
    leaf_lat = key.traits["location1_median"]
    leaf_long = key.traits["location2_median"]

    nodes_backward = parents[key]["path"]
    # print(nodes_backward)
    
    for node in nodes_backward:
        node_dist = calc_dist(node, leaf_lat, leaf_long)
        distance.append(node_dist)
        node_changeLat, node_changeLong = calc_direction(node, leaf_lat, leaf_long)
        changeLat.append(node_changeLat)
        changeLong.append(node_changeLong)

    # print(distance)

    parents[key]["distance"] = distance
    parents[key]["changeLatitude"] = changeLat
    parents[key]["changeLongitude"] = changeLong

In [126]:
print(parents)

{<baltic.baltic.leaf object at 0x32ac94050>: {'path': [<baltic.baltic.node object at 0x32ac8ff90>, <baltic.baltic.node object at 0x32ab06cd0>, <baltic.baltic.node object at 0x32ac8ff50>, <baltic.baltic.node object at 0x32ac8ff10>, <baltic.baltic.node object at 0x32ac8fed0>], 'distance': [289.52441572180527, 879.0984998328062, 836.4205227812602, 1451.7062899210257, 1294.5578746149852], 'changeLatitude': [2.2437204080556725, 4.877985497830139, 5.689164497538883, 11.72288009902357, 10.939295492192478], 'changeLongitude': [1.8410440555472007, 8.481904151825063, 6.671677422858025, 7.51061078654206, 5.257005667340749]}, <baltic.baltic.leaf object at 0x32ac94850>: {'path': [<baltic.baltic.node object at 0x32ac947d0>, <baltic.baltic.node object at 0x32ac8ff90>, <baltic.baltic.node object at 0x32ab06cd0>], 'distance': [150.5461197690163, 170.4315341507949, 779.0506062847204], 'changeLatitude': [-0.2688462615208067, -0.09253574347156501, 2.541729346302901], 'changeLongitude': [1.812276624656917,

In [7]:
# def return_all_leaves(node):
#     leaves[node.label] = node.leaves
#     # no_leaves[node.label] = len(node.leaves)
#     return leaves

In [13]:
# This loop uses the return_all_leaves function to return a dictionary of all leaves per node

parse_count = 0

for k in mytree.Objects:
    if parse_count < 10:
        if k.is_node():
            node = k
            leaves = {}
            # no_leaves = {}
            return_all_leaves(node)
            parse_count += 1
            #print(leaves)

In [87]:
# Function is meant to make a list of all the nodes in the path to each leaf

def return_path_to_leaf(node, leaf, path_to_leaf):
    for child in node.children:
        # print(f"Child: {child}, has leaves: {hasattr(child, 'leaves')}")
        # print(node.children)
        if child == leaf: # If the child is the leaf, stop parsing
            break
        elif hasattr(child, 'leaves') and leaf in child.leaves:  # Optional optimization: only go down relevant branches
            # path_to_leaf.append(child)
            path_to_leaf.append(child.label)
            return_path_to_leaf(child, leaf, path_to_leaf)
        else:
            pass

In [ ]:
parse_count = 0

for k in mytree.Objects:
    if parse_count < 2:
        if k.is_node(): # Parse through each node in the tree
            node = k
            leaves_for_node = []
            leaves_for_node.extend(node.leaves)
            parse_count += 1
            for leaf in leaves_for_node: # print a path to each leaf for each node
                path_to_leaf = []
                return_path_to_leaf(node, leaf, path_to_leaf)
            # print(leaves_for_node[0:3])
            print(path_to_leaf)

In [32]:
# for each node, go 6 months into the future and count how many node and tips have branched from that node

# t = 0.5
# count = 0
# children_count = 0
# child_dict = {}

# for k in mytree.Objects:
#     if count < 10: # only print 10 times
#         key = k.label
#         if k.is_node(): # if an object is a node, then...
#             time = k.length # time is the length of each branch
#             while time < t: # while the time is less than six months
#                 count +=1 # Increase count by one so it knows when to stop counting 
#                 if k.children[0].is_node(): # if the first of K's two children is a node...
#                     children_count += len(k.children) # count the number of childen
#                     if key not in child_dict: # If k is not already in child_dict, add it
#                         child_dict[key] = []
#                     child_dict[key].append(children_count) # Make a dictionary for the number of children in each k
#                     k = k.children[0] # Then name the node child the new k
#                     time += k.length # add the branch length of k to the time (until you get to 6 month)
#                 elif k.children[0].is_leaf(): # If the first child is a 
#                     children_count += len(k.children) # count the number of childen
#                     child_dict = child_dict[key].append(children_count) # Make a dictionary for each
#                 else:
#                     print(f"something unexpected at k {k.label}")

In [ ]:
# Original version of this code (don't want to mess with the original as I try to fix it)

# for k in mytree.Objects:
#     if count >= 10: # only do this 10 times
#         break

#     key = k.label # key is the label (number) of the node
#     time = k.length # time is the length of each branch

#     if k.is_node(): # if k is a leaf:
        
#         # node = k  # use a separate variable to traverse
    
#         while time < t: # while the time is less than six months

#             children_count = len(k.children) # count the number of childen
            
#              # Initialize list if key doesn't exist
#             if key not in child_dict:
#                 child_dict[key] = []
#             child_dict[key].append(children_count) # Make a dictionary for the number of children in each k

            
#             if k.children[0].is_node(): # if the first of K's two children is a node...

#                 children_count += 2
#                 k = k.children[0] # Then name the node child the next node
#                 time += k.length # add the branch length of k to the time (until you get to 6 month)
#                 count += 1 # Increase count by one so it knows when to stop counting
    
#             elif k.children[0].is_leaf(): # If the first child is a leaf

#                 children_count += 1
#                 count += 1 # Increase count by one so it knows when to stop counting
#                 break  # reached leaf, no further traversal
    
#             else:
#                 print(f"something unexpected at node {node.label}")
#                 # break
        
#     elif not k.is_node():
#         child_dict[key].append("not node")

In [26]:
# Second for loop, updated from my attempt by Chat GPT

t = 0.5
count = 0
child_dict = {}

for k in mytree.Objects:
    if count >= 20: # only do this 10 times
        break

    key = k.label # key is the label (number) of the node
    time = k.length # time is the length of each branch

    if k.is_node(): # if k is a leaf:
        
        # node = k  # use a separate variable to traverse
        children_count = 0
    
        while time < t: # while the time is less than six months

            # children_count = len(k.children) # count the number of childen
            
            if k.children[0].is_node(): # if the first of K's two children is a node...

                children_count += 2 # Because a node has two more children...?
                k = k.children[0] # Then name the node child the next node
                time += k.length # add the branch length of k to the time (until you get to 6 month)
                count += 1 # Increase count by one so it knows when to stop counting
    
            elif k.children[0].is_leaf(): # If the first child is a leaf

                children_count += 1 # Because a leaf is one child
                count += 1 # Increase count by one so it knows when to stop counting
                break  # reached leaf, no further traversal
    
            else:
                print(f"something unexpected at node {node.label}")
                # break
        
        # Initialize list if key doesn't exist
        if key not in child_dict:
            child_dict[key] = []
        child_dict[key].append(children_count) # Make a dictionary for the number of children in each k

    elif not k.is_node():
        child_dict[key].append("not node")